# Phase 02 — Competence → Employability

## Step 13 — Final Accuracy Improvement Experiments

This notebook documents the Phase 02 accuracy improvement run using the expanded unprepared dataset `data/raw/phase02_competence_new.csv`.

It repeats the Phase 02 evidence order from notebooks 01–08 and 11:

1. preprocess and validate Likert items
2. reliability
3. construct scores
4. correlation
5. feature sets A (constructs) / B (item-level research)
6. repeated CV model comparison
7. nested CV confirmation
8. freeze only if improved

Notebooks 01–12 remain the original 1000-row development evidence. Phase 01 and Phase 03 files are not modified.

### Previous frozen model
- Linear Regression + StandardScaler on 3 competence constructs
- Test R² = **0.546**, Repeated CV mean R² = **0.538** (1000 rows)

### New dataset
- `phase02_competence_new.csv`: **1500 rows** (266 real, 1234 synthetic)
- Same schema: C1–C10, EVS1–EVS12, data_source
- No exact duplicate rows removed

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "phase2_competence":
    PROJECT_DIR = PROJECT_DIR.parent.parent

RAW_FILE = PROJECT_DIR / "data" / "raw" / "phase02_competence_new.csv"
PREPARED = PROJECT_DIR / "data" / "processed" / "phase2_competence_prepared.csv"
ACCURACY_DIR = PROJECT_DIR / "reports" / "phase2" / "accuracy_improvement"
MODEL_PATH = PROJECT_DIR / "models" / "phase2" / "phase2_competence_final_model.pkl"

print("RAW exists:", RAW_FILE.exists())
print("Prepared exists:", PREPARED.exists())
print("Model exists:", MODEL_PATH.exists())

## 1. Preprocessing summary

Notebook 01 rules applied to the new file:
- validate Response_ID, C1–C10, EVS1–EVS12, data_source
- keep Likert values in 1–5
- remove exact duplicate rows (0 removed)
- compute construct means and EVS_avg target

Saved outputs:
- `data/processed/phase2_competence_prepared.csv`
- `data/processed/phase2_competence_ml_features.csv`

In [ ]:
prepared = pd.read_csv(PREPARED)
print("Rows:", len(prepared))
print(prepared["data_source"].value_counts())
prepared[["LifeCareer_avg", "LearningInnovation_avg", "ITMedia_avg", "EVS_avg"]].describe().round(3)

## 2. Repeated CV model comparison

Feature Set A (deployable): `LifeCareer_avg`, `LearningInnovation_avg`, `ITMedia_avg`

Feature Set B (research only): C1–C10 item level

Models compared: LinearRegression, Ridge, ElasticNet, Huber, SVR, RF, ExtraTrees, GradientBoosting, HistGradientBoosting, XGBoost

In [ ]:
comparison = pd.read_csv(ACCURACY_DIR / "Model_Comparison_RepeatedCV.csv")
comparison[comparison["Deployable"] == True].head(10)

## 3. Nested CV confirmation

Top deployable candidates confirmed with 5-fold nested CV on Feature Set A.

In [ ]:
nested = pd.read_csv(ACCURACY_DIR / "Final_Candidate_Nested_Comparison.csv")
nested

## 4. Final replacement decision

**Linear Regression retained** because nested OOF R² (0.926) was within 0.005 of SVR_RBF (0.927), preserving the Phase 02 interpretability and SHAP evidence architecture from notebooks 09–10.

| Metric | Previous (1000 rows) | New (1500 rows) |
|--------|---------------------|-----------------|
| Nested OOF R² | 0.538 | **0.926** |
| Test R² | 0.546 | **0.930** |
| MAE | 0.182 | **0.179** |
| RMSE | 0.257 | **0.236** |

In [ ]:
decision = pd.read_csv(ACCURACY_DIR / "Final_Model_Replacement_Decision.csv")
decision

## 5. Frozen model artifacts

Updated files:
- `models/phase2/phase2_competence_final_model.pkl`
- `models/phase2/phase2_competence_model.pkl`
- `models/phase2/phase2_competence_model_metadata.json`
- `models/phase2/phase2_competence_freeze_manifest.json`
- `reports/phase2/final_validation/Reference_Predictions.csv`
- `reports/phase2/final_validation/Final_Model_Metrics.csv`

In [ ]:
import json
import joblib

metadata = json.loads((PROJECT_DIR / "models" / "phase2" / "phase2_competence_model_metadata.json").read_text())
print(json.dumps(metadata["metrics"], indent=2))

model = joblib.load(MODEL_PATH)
sample = prepared[["LifeCareer_avg", "LearningInnovation_avg", "ITMedia_avg"]].head(3)
print("Sample predictions:", model.predict(sample).round(3))